# 04 Model: baselines, LASSO, heating sensitivity and small neural networks

Drafted with an AI assistant, see docs/ai_log.md.

Inputs: `data/processed/model_table.parquet`, `data/processed/heating_features.parquet` (sensitivity models only).
Outputs: `data/processed/predictions.parquet` (`egid, cutoff, split, model, score`) and `figures/table_lasso_coef.tex` (written here because it needs the fitted model).

Seven models; each adds one frame to `frames`:
- `roof_yield` (baseline, nothing fitted): yearly electricity yield of the suitable roof surfaces.
- `nbr_share` (baseline, nothing fitted): share of PV buildings among the neighbours within 250 m (`nbr_pv_share_250`; the radius was fixed before any result, see docs/decisions.md).
- `lasso`: L1-penalised logistic regression, fitted on the train split only.
- `lasso_heat_source`, `lasso_heat_date` (sensitivity run, not part of the headline model): the `lasso` pipeline plus heating variables, see the last section.
- `mlp`, `mlp_wide`: small neural networks on the same features as `lasso` (do non-linearities and interactions add anything?), see the last section.

In [ ]:
import time

import helpers
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
model_table = pd.read_parquet(helpers.PROCESSED / "model_table.parquet")
keys = model_table[["egid", "cutoff", "split"]]
frames = []

# Baseline 1: roof yield
frames.append(keys.assign(model="roof_yield", score=model_table["yield_kwh"]))

# Baseline 2: neighbour PV share at 250 m
frames.append(keys.assign(model="nbr_share", score=model_table["nbr_pv_share_250"]))

## LASSO

Everything fitted comes from the train split: decile edges, medians, category levels, means and standard deviations (`helpers.fit_features`), the penalty `C` and the coefficients. The test split is only scored.

Penalty `C` is chosen on a log grid of 10 values by 5-fold cross-validation grouped by municipality (`ggdenr`, `helpers.grouped_cv`), scoring ROC AUC, class weights `balanced`. The best mean AUC decides. The one-standard-error value (smallest `C`, i.e. strongest penalty, within one standard error of the best) is printed for information only. The standardisation is fitted once on the whole train split, not inside each fold (a small, usual shortcut).

In [ ]:
train = model_table[model_table["split"] == "train"].reset_index(drop=True)
test = model_table[model_table["split"] == "test"].reset_index(drop=True)

# Features: fit on train, apply to both
params = helpers.fit_features(train)
x_train = helpers.apply_features(train, params)
x_test = helpers.apply_features(test, params)
y_train = train["y"].to_numpy()
print(x_train.shape[1], "features")


def fit_lasso(x, y, c):
    # l1_ratio=1 is the pure L1 penalty (the `penalty` argument is deprecated)
    model = LogisticRegression(
        l1_ratio=1, C=c, solver="liblinear", class_weight="balanced",
        tol=1e-5, max_iter=1000, random_state=0,
    )  # fmt: skip
    return model.fit(x, y)


# Grouped 5-fold CV over the grid of C (the grid reaches the plateau of the CV curve)
start = time.time()
c_grid = np.logspace(-3, -1, 10)
folds = helpers.grouped_cv(train["ggdenr"], n_splits=5)
cv_rows = []
for c_value in c_grid:
    aucs, n_nonzero = [], []
    for fit_idx, val_idx in folds:
        model = fit_lasso(x_train.iloc[fit_idx], y_train[fit_idx], c_value)
        aucs.append(roc_auc_score(y_train[val_idx], model.decision_function(x_train.iloc[val_idx])))
        n_nonzero.append(int((model.coef_ != 0).sum()))
    cv_rows.append(
        {
            "C": c_value,
            "mean_auc": np.mean(aucs),
            "se_auc": np.std(aucs, ddof=1) / np.sqrt(len(aucs)),
            "mean_nonzero": np.mean(n_nonzero),
        }
    )
cv = pd.DataFrame(cv_rows)
print(cv.round(5).to_string(index=False))

# Best mean AUC is the model used; one-standard-error C is for information only
best = cv["mean_auc"].idxmax()
c_best = cv.loc[best, "C"]
within = cv[cv["mean_auc"] >= cv.loc[best, "mean_auc"] - cv.loc[best, "se_auc"]]
c_1se = within["C"].min()

# Refit on the whole train split
lasso = fit_lasso(x_train, y_train, c_best)
n_nonzero = int((lasso.coef_ != 0).sum())
n_nonzero_1se = int((fit_lasso(x_train, y_train, c_1se).coef_ != 0).sum())
print(f"chosen C {c_best:.5f}: {n_nonzero} of {x_train.shape[1]} coefficients non-zero")
print(f"one-standard-error C {c_1se:.5f}: {n_nonzero_1se} non-zero (information only)")
print(f"fit time {time.time() - start:.0f} s")

In [ ]:
# Coefficient table: non-zero terms on standardised features, largest absolute value first
coef = pd.Series(lasso.coef_[0], index=x_train.columns)
coef = coef[coef != 0].sort_values(key=abs, ascending=False)
lines = [
    r"\begin{tabular}{lr}",
    r"\toprule",
    r"Feature (standardised) & Coefficient \\",
    r"\midrule",
]
for name, value in coef.items():
    lines.append(f"{name.replace('_', chr(92) + '_')} & {value:.3f} \\\\")
lines += [
    r"\midrule",
    r"\multicolumn{2}{l}{"
    + f"$C = {c_best:.4f}$, {n_nonzero} of {x_train.shape[1]} coefficients non-zero, "
    + f"mean CV AUC {cv.loc[best, 'mean_auc']:.4f}"
    + r"} \\",
    r"\bottomrule",
    r"\end{tabular}",
]
(helpers.FIGURES / "table_lasso_coef.tex").write_text("\n".join(lines) + "\n")
print(coef.head(15).round(3).to_string())

Scores are predicted probabilities (with `balanced` class weights they are not calibrated: they over-state the base rate, which is fine for ranking). The `lasso` scores on the train split are in-sample (the model was fitted on those rows); only the test split is out-of-time.

In [ ]:
# LASSO scores: predicted probability, same row order as model_table
score = np.empty(len(model_table))
for split_name, part, x_part in [("train", train, x_train), ("test", test, x_test)]:
    rows = (model_table["split"] == split_name).to_numpy()
    assert (model_table.loc[rows, "egid"].to_numpy() == part["egid"].to_numpy()).all()
    score[rows] = lasso.predict_proba(x_part)[:, 1]
frames.append(keys.assign(model="lasso", score=score))

## Heating sensitivity run (leakage-contaminated upper bound)

Two extra models, fitted exactly like `lasso` (same feature pipeline fitted on train, `balanced` class weights, liblinear, `random_state=0`) at the penalty `C` already chosen for `lasso`. There is no second cross-validation: the CV curve is flat above C = 0.036 (docs/decisions.md), so a second search would double the runtime for no visible gain, and the two models stay comparable to `lasso` at the same penalty.

- `lasso_heat_source`: main features plus one-hot `heat_source` (energy source of the main heating, today's register).
- `lasso_heat_date`: the same plus `heat_updated_after` and `heat_date_missing`.

Heating enters only through the explicit `extra` argument of `helpers.fit_features`; the main `lasso` still refuses heating columns. The heating variables come from today's register snapshot, so `lasso_heat_date` in particular sees information dated after the cut-off. Read its gap to `lasso` as an upper bound contaminated by leakage, not as a gain a real forecaster could have had.

In [ ]:
# Heating features joined on (egid, cutoff); same row order as model_table
heat = pd.read_parquet(helpers.PROCESSED / "heating_features.parquet")
table_h = model_table.merge(heat, on=["egid", "cutoff"], how="left", validate="one_to_one")
assert (table_h["egid"].to_numpy() == model_table["egid"].to_numpy()).all()
assert table_h["heat_source"].notna().all()
heat_columns = ["heat_source", "heat_updated_after", "heat_date_missing"]

heat_sets = {
    "lasso_heat_source": ["heat_source"],
    "lasso_heat_date": ["heat_source", "heat_updated_after", "heat_date_missing"],
}
start = time.time()
heat_coefs = {}
for name, extra in heat_sets.items():
    # Keep only the heating columns this model is allowed to use (others would be refused)
    drop = [c for c in heat_columns if c not in extra]
    table_m = table_h.drop(columns=drop)
    train_h = table_m[table_m["split"] == "train"].reset_index(drop=True)
    test_h = table_m[table_m["split"] == "test"].reset_index(drop=True)
    params_h = helpers.fit_features(train_h, extra=extra)
    x_train_h = helpers.apply_features(train_h, params_h)
    x_test_h = helpers.apply_features(test_h, params_h)
    model_h = fit_lasso(x_train_h, y_train, c_best)  # the C chosen for `lasso`
    score_h = np.empty(len(table_h))
    for split_name, part, x_part in [("train", train_h, x_train_h), ("test", test_h, x_test_h)]:
        rows = (table_h["split"] == split_name).to_numpy()
        assert (table_h.loc[rows, "egid"].to_numpy() == part["egid"].to_numpy()).all()
        score_h[rows] = model_h.predict_proba(x_part)[:, 1]
    frames.append(keys.assign(model=name, score=score_h))
    coef_h = pd.Series(model_h.coef_[0], index=x_train_h.columns)
    is_heat = coef_h.index.str.startswith(("heat_source", "heat_updated", "heat_date"))
    heat_coefs[name] = coef_h[is_heat]
    print(name, x_train_h.shape[1], "features,", int((coef_h != 0).sum()), "non-zero")
    print(heat_coefs[name].round(3).to_string())
print(f"fit time {time.time() - start:.0f} s")

## Small neural networks (do interactions add anything over the LASSO?)

Two scikit-learn `MLPClassifier`s on exactly the `lasso` feature matrix (`x_train`, `x_test`, no heating), fitted on the train split only. Both architectures were fixed before any result and none else is tried; both are reported whatever they give:
- `mlp`: one hidden layer of 16 units.
- `mlp_wide`: two hidden layers (32, 16), to see whether more capacity helps or only overfits.

Common settings: ReLU, Adam, `alpha=1e-4`, `random_state=0`. No tuning.

Stopping rule (train split only, the test split is only scored once both networks are final):
1. Hold out whole municipalities: `GroupShuffleSplit(test_size=0.2, random_state=0)` on `ggdenr` splits the train rows into a fitting part and a validation part.
2. Train epoch by epoch (`partial_fit`) on the fitting part for a fixed budget of 60 epochs, with the ROC AUC on the validation part after each epoch. The best epoch is the one with the highest validation AUC (the first in case of ties). No patience rule.
3. Refit a fresh network with the same settings on the whole train split for exactly that number of epochs. This network gives the train (in-sample) and test scores.

History: the first version used scikit-learn's built-in `early_stopping=True`. It scores a random hold-out by accuracy, which is nearly flat at an 8.6% base rate, so both networks stopped after 12 iterations and were under-fitted (in-sample ROC AUC 0.609 and 0.654 against 0.682 for the LASSO). That was a defect of the stopping rule, visible on the train split alone, so the rule was replaced by the one above; the architectures and all other settings are unchanged.

- `MLPClassifier` has no class weights, so the networks are fitted unweighted. This does not matter for a ranking metric, but their scores are on a different scale from the LASSO's (close to the base rate, not inflated by `balanced` weights).
- The validation part is used only to pick the number of epochs, not to choose between candidates; the held-out municipalities make its AUC an honest spatial hold-out.
- Scores are rounded to 4 decimals (the other models: 6) because neural-network training accumulates floating-point differences across Python and library builds; ties are broken by `egid` in the metric, so the result stays deterministic given identical rounded scores.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neural_network import MLPClassifier


def train_mlp(layers, x_fit, y_fit, n_epochs, x_val=None, y_val=None):
    """Fresh network, one pass per `partial_fit`; returns it and the validation AUC per epoch."""
    net = MLPClassifier(
        hidden_layer_sizes=layers, activation="relu", solver="adam", alpha=1e-4,
        early_stopping=False, random_state=0,
    )  # fmt: skip
    aucs = []
    for _ in range(n_epochs):
        net.partial_fit(x_fit, y_fit, classes=[0, 1])
        if x_val is not None:
            aucs.append(roc_auc_score(y_val, net.predict_proba(x_val)[:, 1]))
    return net, aucs


# Municipality hold-out inside the train split (the test split is not touched here)
fit_idx, val_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(
        x_train, y_train, train["ggdenr"]
    )
)
n_epochs_budget = 60

# MLP architectures declared in advance (docs/decisions.md)
mlp_sets = {"mlp": (16,), "mlp_wide": (32, 16)}
mlp_scores = {}
for name, layers in mlp_sets.items():
    start = time.time()
    _, aucs = train_mlp(
        layers, x_train.iloc[fit_idx], y_train[fit_idx], n_epochs_budget,
        x_train.iloc[val_idx], y_train[val_idx],
    )  # fmt: skip
    best_epoch = int(np.argmax(aucs)) + 1  # argmax takes the first of tied epochs
    net, _ = train_mlp(layers, x_train, y_train, best_epoch)  # refit on the whole train split
    fit_seconds = time.time() - start
    p_train = net.predict_proba(x_train)[:, 1]
    p_test = net.predict_proba(x_test)[:, 1]
    mlp_scores[name] = p_test
    # Score = probability of class 1, same row order as model_table, rounded to 4 decimals
    score_n = np.empty(len(model_table))
    for split_name, part, p_part in [("train", train, p_train), ("test", test, p_test)]:
        rows = (model_table["split"] == split_name).to_numpy()
        assert (model_table.loc[rows, "egid"].to_numpy() == part["egid"].to_numpy()).all()
        score_n[rows] = p_part
    frames.append(keys.assign(model=name, score=score_n.round(4)))
    curve = {e: round(aucs[e - 1], 4) for e in (1, 5, 10, 20, 30, 40, 50, 60)}
    print(f"{name} {layers}: validation AUC by epoch {curve}")
    print(
        f"  best epoch {best_epoch}, validation AUC {aucs[best_epoch - 1]:.4f}, "
        f"fit time {fit_seconds:.1f} s"
    )
    print(
        f"  ROC AUC train (in-sample) {roc_auc_score(y_train, p_train):.4f}, "
        f"capture 10% train {helpers.capture_at_k(y_train, p_train, 0.10):.4f}, "
        f"test {roc_auc_score(test['y'], p_test):.4f}"
    )


Fragility check: how many test buildings would enter or leave the top 10% if the (rounded) scores moved by up to 1e-4 (uniform noise, seed 0)? A large count would mean the capture metric of the networks depends on rounding ties.

In [ ]:
rng = np.random.default_rng(0)
y_test = test["y"].astype(int).to_numpy()
n_top = int(np.ceil(0.10 * len(test)))
for name, p_test in mlp_scores.items():
    base = np.round(p_test, 4)
    moved = base + rng.uniform(-1e-4, 1e-4, len(base))
    top_base = np.zeros(len(base), bool)
    top_base[np.argsort(-base, kind="stable")[:n_top]] = True
    top_moved = np.zeros(len(base), bool)
    top_moved[np.argsort(-moved, kind="stable")[:n_top]] = True
    changed = int((top_base != top_moved).sum())
    cap_base = helpers.capture_at_k(y_test, base, 0.10)
    cap_moved = helpers.capture_at_k(y_test, moved, 0.10)
    print(
        f"{name}: {changed} buildings change top-10% membership "
        f"({n_top} in top 10%); capture_10 {cap_base:.4f} -> {cap_moved:.4f}"
    )

In [ ]:
# One table, sorted deterministically; scores as float64 (networks already at 4 decimals)
predictions = pd.concat(frames, ignore_index=True)
predictions["score"] = predictions["score"].astype("float64").round(6)  # no-op for the networks
predictions = predictions.sort_values(["cutoff", "model", "egid"]).reset_index(drop=True)
print(predictions["score"].isna().sum(), "missing scores")
print(predictions.groupby(["model", "split"]).size())
predictions.to_parquet(helpers.PROCESSED / "predictions.parquet", index=False)